In [21]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import os, sys
sys.path.append("/home/akapociu/ift/interactiondynamics")

# TODO: set this to your repo root if needed
REPO_ROOT = Path.cwd()
PKG_ROOT = REPO_ROOT / "interactiondynamics"

if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

from datasets.wave_1d import WaveEquationBinnedDataset, WaveEquationBinnedConfig
from datasets.spring_web_2d import SpringWeb2DDataset, SpringWeb2DConfig
from datasets.charged_particles import (
    ChargedParticlesBinnedDataset,
    ChargedParticlesBinnedConfig,
    _pair_record as charged_pair_record,
    _total_energy as charged_total_energy,
)

In [22]:
TARGET_PCTS = list(range(10, 100, 10))  # 10,20,...,90

DATASET_METRIC_OPTIONS = {
    "wave": ["pair_accel", "rel_q", "rel_v", "pair_grad"],
    "spring_web": ["force_mag", "extension", "distance", "rel_speed"],
    "charged_particles": ["force_threshold", "distance_threshold"],
}

# -------------------------
# choose what to analyze
# -------------------------
DATASET_NAME = "charged_particles"              # "wave", "spring_web", "charged_particles"
METRIC_NAME = "force_threshold"         # see DATASET_METRIC_OPTIONS above
SPLIT = "all"                      # "all", "train", "val", "test"

# used only for wave / spring_web
USE_ABS = True

# optional dataset-specific overrides
# leave empty {} unless you want to match a particular variant
CFG_OVERRIDES = {}

print("Available metrics:")
for k, v in DATASET_METRIC_OPTIONS.items():
    print(f"{k}: {v}")

Available metrics:
wave: ['pair_accel', 'rel_q', 'rel_v', 'pair_grad']
spring_web: ['force_mag', 'extension', 'distance', 'rel_speed']
charged_particles: ['force_threshold', 'distance_threshold']


In [23]:
def _resolve_wave_indices(ds, split):
    if split == "all":
        return list(range(len(ds._all_bins)))
    b0, b1 = ds._split_ranges[split]
    return list(range(b0, b1 + 1))


def _resolve_charged_indices(ds, split):
    if split == "all":
        return list(range(ds.cfg.num_bins))
    b0, b1 = ds._split_ranges[split]
    return list(range(b0, b1 + 1))


def _resolve_springweb_indices(ds, split):
    if split == "all":
        return list(range(ds.cfg.num_bins))
    return list(ds.split_bins[split])


def collect_wave_candidate_values(metric="pair_accel", use_abs=True, split="all", cfg_overrides=None):
    cfg_overrides = cfg_overrides or {}

    cfg = WaveEquationBinnedConfig(
        event_mode="all_neighbors",   # inspect all candidate edges before thresholding
        threshold_metric=metric,
        threshold_use_absolute=use_abs,
        threshold_keep_one_if_empty=True,
        **cfg_overrides,
    )
    ds = WaveEquationBinnedDataset(cfg)

    q_traj, v_traj, _ = ds.trajectory()
    t_idxs = _resolve_wave_indices(ds, split)

    dx = cfg.domain_length / float(cfg.num_nodes)
    c = cfg.wave_speed

    vals_by_bin = []

    for t_idx in t_idxs:
        q = q_traj[t_idx]
        v = v_traj[t_idx]
        vals = []

        for recv in range(cfg.num_nodes):
            left = (recv - 1) % cfg.num_nodes
            right = (recv + 1) % cfg.num_nodes

            for send in [left, right]:
                rel_q = float(q[send] - q[recv])
                rel_v = float(v[send] - v[recv])
                pair_grad = rel_q / dx
                pair_accel = (c ** 2) * rel_q / (dx ** 2)

                score = {
                    "pair_accel": pair_accel,
                    "rel_q": rel_q,
                    "rel_v": rel_v,
                    "pair_grad": pair_grad,
                }[metric]

                vals.append(abs(score) if use_abs else score)

        vals_by_bin.append(np.asarray(vals, dtype=float))

    return vals_by_bin, "ge", 1, cfg


def collect_springweb_candidate_values(metric="force_mag", use_abs=True, split="all", cfg_overrides=None):
    cfg_overrides = cfg_overrides or {}

    cfg = SpringWeb2DConfig(
        event_mode="all_neighbors",   # inspect all candidate edges before thresholding
        threshold_metric=metric,
        threshold_use_absolute=use_abs,
        threshold_keep_one_if_empty=True,
        **cfg_overrides,
    )
    ds = SpringWeb2DDataset(cfg)

    chosen_bins = set(_resolve_springweb_indices(ds, split))

    n = cfg.num_nodes
    dt = cfg.dt
    k = cfg.spring_k
    damping = cfg.damping

    g = torch.Generator().manual_seed(cfg.seed)

    x0 = ds._ideal_ring_positions(n, cfg.ring_radius)
    spring_pairs = ds._build_spring_pairs(x0)

    x = x0 + cfg.init_pos_noise * torch.randn(n, 2, generator=g)
    v = cfg.init_vel_noise * torch.randn(n, 2, generator=g)

    vals_by_bin = []

    for b in range(cfg.num_bins):
        net_force = torch.zeros(n, 2, dtype=torch.float32)
        vals = []

        for i, j, rest in spring_pairs:
            dpos = x[j] - x[i]
            dvel = v[j] - v[i]

            dist = torch.norm(dpos).clamp_min(1e-8)
            direction = dpos / dist
            extension = dist - rest
            force_vec = k * extension * direction
            force_mag = torch.norm(force_vec)
            rel_speed = torch.norm(dvel)

            score = {
                "force_mag": float(force_mag.item()),
                "extension": float(extension.item()),
                "distance": float(dist.item()),
                "rel_speed": float(rel_speed.item()),
            }[metric]

            score = abs(score) if use_abs else score

            vals.append(score)
            if cfg.bidirectional:
                vals.append(score)

            net_force[i] += force_vec
            net_force[j] -= force_vec

        if b in chosen_bins:
            vals_by_bin.append(np.asarray(vals, dtype=float))

        a = net_force
        v_next = damping * (v + dt * a)
        x_next = x + dt * v_next
        x = x_next
        v = v_next

    return vals_by_bin, "ge", 1, cfg


def collect_charged_candidate_values(metric="force_threshold", split="all", cfg_overrides=None):
    cfg_overrides = cfg_overrides or {}

    cfg = ChargedParticlesBinnedConfig(
        interaction_rule="all_pairs",   # inspect all candidate directed pairs
        obs_edge_keep_prob=1.0,
        distance_threshold_jitter_std=0.0,
        force_threshold_jitter_std=0.0,
        min_edges_per_bin=1,
        **cfg_overrides,
    )
    ds = ChargedParticlesBinnedDataset(cfg)

    loc_traj, vel_traj, charges, _ = ds.trajectory()
    t_idxs = _resolve_charged_indices(ds, split)

    vals_by_bin = []

    for t_idx in t_idxs:
        loc = loc_traj[t_idx]
        vel = vel_traj[t_idx]

        global_energy = charged_total_energy(
            loc,
            vel,
            charges,
            cfg.interaction_strength,
            cfg.softening,
        )

        vals = []

        for recv in range(cfg.num_nodes):
            for send in range(cfg.num_nodes):
                if send == recv:
                    continue

                rec = charged_pair_record(
                    loc,
                    vel,
                    charges,
                    send,
                    recv,
                    cfg,
                    global_energy,
                )

                if metric == "force_threshold":
                    vals.append(float(rec["force_mag"]))
                elif metric == "distance_threshold":
                    vals.append(float(rec["distance"]))
                else:
                    raise ValueError(f"Unknown charged metric: {metric}")

        vals_by_bin.append(np.asarray(vals, dtype=float))

    # force_threshold keeps >= threshold
    # distance_threshold keeps <= threshold
    keep_rule = "ge" if metric == "force_threshold" else "le"

    return vals_by_bin, keep_rule, max(1, int(cfg.min_edges_per_bin)), cfg


def summarize_thresholds(vals_by_bin, keep_rule, min_edges_after_fallback=1, target_pcts=TARGET_PCTS):
    all_vals = np.concatenate(vals_by_bin)
    rows = []

    for pct in target_pcts:
        keep_frac_target = pct / 100.0
        quantile = 1.0 - keep_frac_target if keep_rule == "ge" else keep_frac_target
        thr = float(np.quantile(all_vals, quantile))

        raw_kept_counts = []
        final_kept_counts = []

        for vals in vals_by_bin:
            if keep_rule == "ge":
                raw_kept = int((vals >= thr).sum())
            elif keep_rule == "le":
                raw_kept = int((vals <= thr).sum())
            else:
                raise ValueError(f"Unknown keep_rule: {keep_rule}")

            final_kept = raw_kept
            if raw_kept < min_edges_after_fallback:
                final_kept = min(len(vals), min_edges_after_fallback)

            raw_kept_counts.append(raw_kept)
            final_kept_counts.append(final_kept)

        raw_kept_counts = np.asarray(raw_kept_counts)
        final_kept_counts = np.asarray(final_kept_counts)

        candidate_edges = int(len(vals_by_bin[0]))

        rows.append(
            {
                "target_pct_kept": pct,
                "threshold": thr,
                "candidate_edges_per_bin": candidate_edges,

                "raw_mean_edges_kept_per_bin": float(raw_kept_counts.mean()),
                "raw_mean_pct_kept": 100.0 * float(raw_kept_counts.mean()) / candidate_edges,
                "raw_median_pct_kept": 100.0 * float(np.median(raw_kept_counts)) / candidate_edges,
                "raw_min_pct_kept": 100.0 * float(raw_kept_counts.min()) / candidate_edges,
                "raw_max_pct_kept": 100.0 * float(raw_kept_counts.max()) / candidate_edges,
                "zero_bins_before_fallback": int((raw_kept_counts == 0).sum()),

                "emitted_mean_edges_kept_per_bin": float(final_kept_counts.mean()),
                "emitted_mean_pct_kept": 100.0 * float(final_kept_counts.mean()) / candidate_edges,
                "emitted_min_pct_kept": 100.0 * float(final_kept_counts.min()) / candidate_edges,
                "emitted_max_pct_kept": 100.0 * float(final_kept_counts.max()) / candidate_edges,

                "num_bins_used": len(vals_by_bin),
            }
        )

    return pd.DataFrame(rows)


def get_threshold_table(
    dataset_name,
    metric_name,
    *,
    split="all",
    use_abs=True,
    cfg_overrides=None,
    target_pcts=TARGET_PCTS,
):
    cfg_overrides = cfg_overrides or {}

    if dataset_name == "wave":
        vals_by_bin, keep_rule, min_edges_after_fallback, cfg = collect_wave_candidate_values(
            metric=metric_name,
            use_abs=use_abs,
            split=split,
            cfg_overrides=cfg_overrides,
        )
    elif dataset_name == "spring_web":
        vals_by_bin, keep_rule, min_edges_after_fallback, cfg = collect_springweb_candidate_values(
            metric=metric_name,
            use_abs=use_abs,
            split=split,
            cfg_overrides=cfg_overrides,
        )
    elif dataset_name == "charged_particles":
        vals_by_bin, keep_rule, min_edges_after_fallback, cfg = collect_charged_candidate_values(
            metric=metric_name,
            split=split,
            cfg_overrides=cfg_overrides,
        )
    else:
        raise ValueError(f"Unknown dataset_name: {dataset_name}")

    out = summarize_thresholds(
        vals_by_bin,
        keep_rule=keep_rule,
        min_edges_after_fallback=min_edges_after_fallback,
        target_pcts=target_pcts,
    )

    out.insert(0, "dataset", dataset_name)
    out.insert(1, "metric", metric_name)
    out.insert(2, "split", split)
    out.insert(3, "use_abs", use_abs if dataset_name != "charged_particles" else np.nan)

    return out, cfg

In [24]:
table, cfg = get_threshold_table(
    DATASET_NAME,
    METRIC_NAME,
    split=SPLIT,
    use_abs=USE_ABS,
    cfg_overrides=CFG_OVERRIDES,
)

print("CONFIG USED:")
print(cfg)
print("\nRESULTS:")
display(table)

# optional: grab just the target -> threshold mapping
threshold_lookup = table[["target_pct_kept", "threshold"]].copy()
print("\nJust the thresholds:")
display(threshold_lookup)

CONFIG USED:
ChargedParticlesBinnedConfig(name='charged_particles', num_nodes=64, num_bins=512, split_fracs=(0.7, 0.15, 0.15), box_size=4.0, loc_std=1.0, vel_norm=0.5, interaction_strength=1.0, softening=0.1, micro_dt=0.001, steps_per_bin=250, max_force_clip=100.0, charge_types=(-1.0, 0.0, 1.0), charge_probs=(0.4, 0.2, 0.4), observation_noise_loc=0.0, observation_noise_vel=0.0, distance_threshold_jitter_std=0.0, force_threshold_jitter_std=0.0, obs_edge_keep_prob=1.0, interaction_rule='all_pairs', distance_threshold=2.5, force_threshold=0.2, top_k=64, min_edges_per_bin=1, threshold_splits=('train', 'val', 'test'), seed=0, device=None)

RESULTS:


,dataset,metric,split,use_abs,target_pct_kept,threshold,candidate_edges_per_bin,raw_mean_edges_kept_per_bin,raw_mean_pct_kept,raw_median_pct_kept,raw_min_pct_kept,raw_max_pct_kept,zero_bins_before_fallback,emitted_mean_edges_kept_per_bin,emitted_mean_pct_kept,emitted_min_pct_kept,emitted_max_pct_kept,num_bins_used
0,charged_particles,force_threshold,all,NaN,10,0.304644,4032,403.203125,10.000078,9.672619,7.093254,39.236111,0,403.203125,10.000078,7.093254,39.236111,512
1,charged_particles,force_threshold,all,NaN,20,0.130677,4032,806.402344,20.000058,19.444444,15.525794,60.317460,0,806.402344,20.000058,15.525794,60.317460,512
2,charged_particles,force_threshold,all,NaN,30,0.077757,4032,1209.601562,30.000039,29.315476,21.676587,68.501984,0,1209.601562,30.000039,21.676587,68.501984,512
3,charged_particles,force_threshold,all,NaN,40,0.051782,4032,1612.800781,40.000019,39.533730,30.654762,70.436508,0,1612.800781,40.000019,30.654762,70.436508,512
4,charged_particles,force_threshold,all,NaN,50,0.036162,4032,2016.000000,50.000000,49.751984,39.384921,70.932540,0,2016.000000,50.000000,39.384921,70.932540,512
5,charged_particles,force_threshold,all,NaN,60,0.025295,4032,2419.199219,59.999981,59.895833,51.934524,70.982143,0,2419.199219,59.999981,51.934524,70.982143,512
6,charged_particles,force_threshold,all,NaN,70,0.014224,4032,2822.402344,70.000058,70.089286,67.857143,70.982143,0,2822.402344,70.000058,67.857143,70.982143,512
7,charged_particles,force_threshold,all,NaN,80,0.000000,4032,4032.000000,100.000000,100.000000,100.000000,100.000000,0,4032.000000,100.000000,100.000000,100.000000,512
8,charged_particles,force_threshold,all,NaN,90,0.000000,4032,4032.000000,100.000000,100.000000,100.000000,100.000000,0,4032.000000,100.000000,100.000000,100.000000,512



Just the thresholds:


,target_pct_kept,threshold
0,10,0.304644
1,20,0.130677
2,30,0.077757
3,40,0.051782
4,50,0.036162
5,60,0.025295
6,70,0.014224
7,80,0.000000
8,90,0.000000
